# Low Fidelity Neural Network
Neural Network which uses LF dataset and exploits the Low fidelity component of the MF model. The NN is tested also on the HF test set  

In [ ]:
#########################     LIBRARIES     ##########################
import keras.backend as K
from keras.regularizers import l2
from keras.utils import custom_object_scope
from keras.initializers import glorot_uniform
from keras.models import load_model, save_model
from hyperopt import STATUS_OK, tpe, Trials, hp, fmin
from hyperopt.pyll.stochastic import sample
from sklearn.model_selection import KFold
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import cm
from matplotlib.ticker import LinearLocator, FormatStrFormatter
from keras.optimizers import Adam, Nadam, Adamax
from ann_functions3D import (
    getModel,
    kCrossVal,
    transfBestparam,
    kCrossValSingle,
    import_data,
)
from time import perf_counter
import pandas
import os
from itertools import product
seed = 7
np.random.seed(seed)

# Data Preparation

In [ ]:
file_path_LF = "../DATA/reaction_diffusion_LF_46_d75.mat"
(reaction_LF_test, U_LF_test, x_LF_test)  = import_data(file_path_LF)

In [ ]:
reaction_max = np.max(reaction_LF_test)
reaction_min = np.min(reaction_LF_test)

reaction_LF_test = (reaction_LF_test - reaction_min) / (
    reaction_max - reaction_min
)

In [ ]:
NepoLF=5000
Nlf=30

In [ ]:
permutation1 = np.random.permutation(len(reaction_LF_test))
permutation2 = np.random.permutation(len(x_LF_test))
reaction_LF = reaction_LF_test[permutation1][0:Nlf]
x_LF = x_LF_test[permutation2][0:Nlf]
reaction_LF = np.column_stack((reaction_LF, x_LF))
U_LF_test = U_LF_test[:, -1, :,44]


reaction_LF_test = np.array(list(product(reaction_LF_test.flatten(), x_LF_test.flatten())))

# TRANSFORMATION
U_h_max_test = np.max(U_LF_test)
U_h_min_test = np.min(U_LF_test)
#U_HF = (U_HF - U_h_min_test) / (U_h_max_test - U_h_min_test)
U_LF_test = (U_LF_test - U_h_min_test) / (U_h_max_test - U_h_min_test)

U_LF = U_LF_test[permutation1[0:Nlf],permutation2[0:Nlf]]
row, col = U_LF_test.shape
index_row, index_col = np.meshgrid(np.arange(row), np.arange(col), indexing='ij')
comb = np.ravel_multi_index((index_row.flatten(), index_col.flatten()), dims=(row, col))
U_LF_test = U_LF_test.flatten()[comb]

# Low Fidelity Neural Network
## Low fidelity data

In [ ]:
start = perf_counter()

reaction_final = np.vstack(
    (reaction_LF)
)  

##########################       NN_LF     ##########################
####################    HYPERPARAMETER OPTIMIZATION    #######################
MAX_EVAL = 15
name = "LF"
K.clear_session()
bayes_trials = Trials()
opt_list = ["Adam", "Adamax"]
kernel_list = ["uniform", "glorot_uniform"]
aux_dic = {"opt": opt_list, "kernel_init": kernel_list}
space = {
    "nodes": hp.qloguniform("nodes", np.log(4), np.log(64), 2),
    "l2weight": hp.loguniform("l2weight", np.log(0.0001), np.log(100)),
    "lr": hp.loguniform("lr", np.log(0.0001), np.log(0.1)),
    "kernel_init": hp.choice("kernel_init", kernel_list),
    "opt": hp.choice("opt", opt_list),
}


def objective(params):
    K.clear_session()
    CVres = kCrossValSingle(Nlf, NepoLF, reaction_final, U_LF, params, name)
    return {"loss": CVres, "params": params, "status": STATUS_OK}


best_params = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=MAX_EVAL,
    trials=bayes_trials,
)

transfBestparam(best_params, aux_dic)
print(best_params)

####################    NN training and PREDICTION    #######################
start2 = perf_counter()
finalModel = getModel(
    best_params, name
)  # final model chosen according to the best paramters
hist = finalModel.fit(
    reaction_LF,
    U_LF,
    validation_data=(reaction_LF_test, U_LF_test),
    epochs=NepoLF,
    batch_size=Nlf,
    verbose=0,
    validation_freq=20,
)

ULF = finalModel.predict(reaction_LF_test)

stop = perf_counter()
elapsed = stop - start
print("Elapsed time: ", elapsed)
print("\nLF Model:")

elapsed2 = stop - start2
print("Elapsed time: ", elapsed2)
print("\nLF Model:")

test_mse = np.mean(np.square(U_LF_test - ULF[:, 0]))
print(f"Test MSE: {test_mse}")

r2_LF = 1 - np.sum(np.square(U_LF_test - ULF[:, 0])) / np.sum(
    np.square(U_LF_test - np.mean(U_LF_test))
)
print(f"R^2: {r2_LF}")

# High fidelity data

# da qui check

In [ ]:
####################    NN training and PREDICTION HF   #######################
# start2 = perf_counter()
finalModel = getModel(
    best_params, name
)  # final model chosen according to the best paramters

file_path_HF = "../DATA/reaction_diffusion_HF.mat"
(reaction_HF_test, U_HF_test) = import_data(file_path_HF)
#U_HF_test = U_HF_test[
#    :, -1, int(np.shape(U_HF_test)[2] / 2), int(np.shape(U_HF_test)[3] / 2)
#]
U_HF_test = U_HF_test[
    :, -1, 44,44
]

reaction_HF_test = (reaction_HF_test - np.min(reaction_HF_test)) / (
    np.max(reaction_HF_test) - np.min(reaction_HF_test)
)
U_HF_test = (U_HF_test - np.min(U_HF_test)) / (
    np.max(U_HF_test) - np.min(U_HF_test)
)

hist = finalModel.fit(
    reaction_LF,
    U_LF,
    validation_data=(reaction_HF_test, U_HF_test),
    epochs=NepoLF,
    batch_size=Nlf,
    verbose=0,
    validation_freq=20,
)

UHF = finalModel.predict(reaction_HF_test)

stop = perf_counter()
elapsed = stop - start
print("Elapsed time: ", elapsed)
print("\nHF Data")
print("\nLF Model:")

elapsed2 = stop - start2
print("Elapsed time: ", elapsed2)
print("\nLF Model:")

test_mse_HF = np.mean(np.square(U_HF_test - UHF[:, 0]))
print(f"Test MSE: {test_mse_HF}")

r2_HF = 1 - np.sum(np.square(U_HF_test - UHF[:, 0])) / np.sum(
    np.square(U_HF_test - np.mean(U_HF_test))
)
print(f"R^2: {r2_HF}")